In [ ]:
%load_ext autoreload
%autoreload 2

import os
import matplotlib.pyplot as plt
import numpy as np

from scipy.constants import R

from mg_sampling import PROJECT_DIR, FORCEFIELDS
from mg_sampling.classes import KineticSims
from mg_sampling.plot import LABELS
from mg_sampling.tools import edges_to_range
from mg_sampling.constants import RT, SWITCH_COEFF

from scipy.integrate import cumulative_trapezoid
from scipy.optimize import minimize_scalar

## Optimize linear ramp

In [ ]:
forcefield = "mMg_tip3p"
path = PROJECT_DIR / 'kinetics' / f"op_{forcefield}"
sims = KineticSims(path)

In [ ]:
cv = np.concatenate([sim.colvar['cv.tot'] for sim in sims if 0.8 <= float(sim.scaling) <= 1.2])
scalings = np.array([float(sim.scaling) for sim in sims if 0.8 <= float(sim.scaling) <= 1.2])
bias = cv[:, None] * scalings / RT
dop1 = np.concatenate([sim.colvar['dop1'] for sim in sims if 0.8 <= float(sim.scaling) <= 1.2])
nop = np.concatenate([sim.colvar['cv.nb-0'] for sim in sims if 0.8 <= float(sim.scaling) <= 1.2])*SWITCH_COEFF

In [ ]:
for i, s in enumerate(scalings):
    plt.scatter(range(len(bias)), bias[:, i], s=0.1)

In [ ]:
from bussilab.wham import wham
W = wham(bias)

In [ ]:
plt.plot(W.logW, ls='none', marker='.', ms=0.1)

In [ ]:
d_edges = np.linspace(0, 1, 100)
d_range = edges_to_range(d_edges)
λs = np.linspace(0, 1, 20)
ΔΔGs = []
for λ in λs:
    u = λ * cv / RT
    weights = np.exp(W.logW - u)
    ρd, _ = np.histogram(dop1, bins=d_edges, weights=weights, density=True)
    plt.plot(d_range, ρd, label=f"{λ:.2f}", color=plt.cm.plasma(0.5*λ))
plt.ylabel(r"$ρ(d)$ [nm⁻¹]")
plt.xlabel(r"$d$ [nm]")
plt.legend(fontsize=8)

In [ ]:
from scipy.optimize import minimize_scalar, minimize

λs = np.linspace(0, 1, 20)

def ramp(x, a):
    return a * x

cutoff = 0.4
p0_bound = np.exp(W.logW)[dop1 < cutoff].sum()
ΔG0 = -RT * np.log(p0_bound / (1 - p0_bound))

u = λs * cv[:, None] / RT
logW0 = W.logW[:, None] - u

def loss(a):
    weights = np.exp(logW0[:, -1] - ramp(nop, a)/RT)
    weights /= weights.sum()
    p_bound = weights[dop1 < cutoff].sum()
    ΔΔG = -RT * np.log(p_bound / (1 - p_bound)) - ΔG0
    return ΔΔG**2

def loss_all(As):
    weights = np.exp(logW0 - ramp(nop[:, None], As)/RT)
    weights /= weights.sum(axis=0)
    p_bound = weights[dop1 < cutoff].sum(axis=0)
    ΔΔG = -RT * np.log(p_bound / (1 - p_bound)) - ΔG0
    return np.mean(ΔΔG**2)

res = minimize_scalar(loss)
res_all = minimize(loss_all, x0=np.zeros_like(λs))

ΔΔGs = []
ΔΔGopts = []

for λ in λs:
    u = λ * cv / RT
    weights = np.exp(W.logW - u)
    weights /= weights.sum()
    p_bound = weights[dop1 < cutoff].sum()
    ΔΔG = -RT * np.log(p_bound / (1 - p_bound)) - ΔG0
    ΔΔGs.append(ΔΔG)

    weights = np.exp(W.logW - u - λ*ramp(nop, res.x)/RT)
    weights /= weights.sum()
    p_bound = weights[dop1 < cutoff].sum()
    ΔΔGopt = -RT * np.log(p_bound / (1 - p_bound)) - ΔG0
    ΔΔGopts.append(ΔΔGopt)

In [ ]:
plt.plot(λs, ΔΔGs, label='Ref.')
plt.plot(λs, ΔΔGopts, label=f'Corrected with ramp (λ = 1)', ls='dotted')

weights = np.exp(logW0 - ramp(nop[:, None], res_all.x)/RT)
weights /= weights.sum(axis=0)
p_bound = weights[dop1 < cutoff].sum(axis=0)
ΔΔGopts_all = -RT * np.log(p_bound / (1 - p_bound)) - ΔG0
plt.plot(λs, ΔΔGopts_all, label=f'Corrected with (each λ)', ls='dashed')
plt.ylabel(r"$ΔΔG_\text{bind}(λ)$ [kJ/mol]")
plt.xlabel(r"λ")
plt.legend()

In [ ]:
scalings = np.loadtxt(PROJECT_DIR / 'replex' / '7ez0' / forcefield / 'scalings.01.txt')
λs = scalings

def ramp(x, a):
    return a * x

cutoff = 0.4
p0_bound = np.exp(W.logW)[dop1 < cutoff].sum()
ΔG0 = -RT * np.log(p0_bound / (1 - p0_bound))

u = λs * cv[:, None] / RT
logW0 = W.logW[:, None] - u

def loss(a):
    weights = np.exp(logW0[:, -1] - ramp(nop, a)/RT)
    weights /= weights.sum()
    p_bound = weights[dop1 < cutoff].sum()
    ΔΔG = -RT * np.log(p_bound / (1 - p_bound)) - ΔG0
    return ΔΔG**2

def loss_all(As):
    weights = np.exp(logW0 - ramp(nop[:, None], As)/RT)
    weights /= weights.sum(axis=0)
    p_bound = weights[dop1 < cutoff].sum(axis=0)
    ΔΔG = -RT * np.log(p_bound / (1 - p_bound)) - ΔG0
    return np.mean(ΔΔG**2)

res = minimize_scalar(loss)
res_all = minimize(loss_all, x0=np.zeros_like(λs))

ΔΔGs = []
ΔΔGopts = []

for λ in λs:
    u = λ * cv / RT
    weights = np.exp(W.logW - u)
    weights /= weights.sum()
    p_bound = weights[dop1 < cutoff].sum()
    ΔΔG = -RT * np.log(p_bound / (1 - p_bound)) - ΔG0
    ΔΔGs.append(ΔΔG)

    weights = np.exp(W.logW - u - λ*ramp(nop, res.x)/RT)
    weights /= weights.sum()
    p_bound = weights[dop1 < cutoff].sum()
    ΔΔGopt = -RT * np.log(p_bound / (1 - p_bound)) - ΔG0
    ΔΔGopts.append(ΔΔGopt)

plt.plot(λs, ΔΔGs, label='Ref.')
plt.plot(λs, ΔΔGopts, label=f'Corrected with ramp (λ = 1)', ls='dotted')

weights = np.exp(logW0 - ramp(nop[:, None], res_all.x)/RT)
weights /= weights.sum(axis=0)
p_bound = weights[dop1 < cutoff].sum(axis=0)
ΔΔGopts_all = -RT * np.log(p_bound / (1 - p_bound)) - ΔG0
plt.plot(λs, ΔΔGopts_all, label=f'Corrected with (each λ)', ls='dashed')
plt.ylabel(r"$ΔΔG_\text{bind}(λ)$ [kJ/mol]")
plt.xlabel(r"λ")
plt.legend()

In [ ]:
ramp_scalings = res_all.x
ramp_scalings[0] = 0.0
np.savetxt(PROJECT_DIR / 'replex' / '7ez0' / forcefield / 'ramp_scalings.01.txt', ramp_scalings)

In [ ]:
from mg_sampling.bias import bias_fn
from mg_sampling.tools import read_wsv
params = read_wsv(f'{sims.path}/bias_params.wsv').iloc[0]
x_range = np.linspace(0, 2, 100)
y_range = np.linspace(4, 7, 100)
xx, yy = np.meshgrid(x_range, y_range, indexing='ij')
extent = [*x_range[[0, -1]], *y_range[[0, -1]]]
plt.imshow(bias_fn(xx, yy, *params).T, extent=extent, origin='lower')


In [ ]:
from matplotlib.colors import CenteredNorm
x_range = np.linspace(0, 2, 100)
y_range = np.linspace(4, 7, 100)
xx, yy = np.meshgrid(x_range, y_range, indexing='ij')
extent = [*x_range[[0, -1]], *y_range[[0, -1]]]
plt.imshow(bias_fn(xx, yy, *params).T + ramp(xx, res.x).T, cmap='coolwarm', extent=extent, origin='lower', norm=CenteredNorm())
plt.colorbar(label=r"$u_\text{bias}$ [kJ/mol]")
plt.xlabel(r"$n_\mathrm{O_P}$")
plt.ylabel(r"$n_\mathrm{O_W}$")

In [ ]:
params

### Load kinetic simulation

### Estimate from Kinetic simulations

In [ ]:
forcefields = ["mMg_tip3p", "nMg_tip3p", "mMg_opc", "ecc_spce"]

fig, axarr = plt.subplots(nrows=len(forcefields), figsize=(6, 3*len(forcefields)))
aopts = []
for ax, forcefield in zip(axarr, forcefields):
    path = PROJECT_DIR / 'kinetics' / f'op_{forcefield}'
    sim = KineticSims(path / '1.0')
    nop = sim.colvar['cv.nb-0']
    d = sim.colvar['dop1']
    bias = sim.colvar['bias.bias']
    d_edges = np.linspace(0, 1, 80)
    ρd, _ = np.histogram(d, bins=d_edges, density=True)
    Fd = -RT*np.log(ρd)
    Fd -= Fd[-1]
    l, = ax.plot(edges_to_range(d_edges), Fd, label=forcefield)
    weights = np.exp(bias/RT)
    ρd, _ = np.histogram(d, bins=d_edges, density=True, weights=weights)
    Fd = -RT*np.log(ρd)
    Fd -= Fd[-1]
    ax.plot(edges_to_range(d_edges), Fd, label=forcefield, ls='dashed')
    
    mask = nop > 0.001
    ΔΔG = RT * np.log(np.average(np.exp(bias/RT), weights=mask.astype(float)))
    print(forcefield, ΔΔG, "kJ/mol")
    def ramp(x, a):
        return a * x
    def loss(a):
        new_bias = bias + ramp(nop, a)
        new_weights = np.exp(-ramp(nop, a)/RT)
        ΔΔG = RT * np.log(np.average(np.exp(new_bias/RT), weights=new_weights*mask))
        return ΔΔG**2
    aopt = minimize_scalar(loss).x
    aopts.append(aopt)
    new_bias = bias + ramp(nop, aopt)
    new_weights = np.exp(-ramp(nop, aopt)/RT)
    ΔΔG = RT * np.log(np.average(np.exp(new_bias/RT), weights=new_weights*mask))
    print(forcefield, f"{aopt = } kJ/mol", f"{ΔΔG = } kJ/mol")
    ρd, _ = np.histogram(d, bins=d_edges, density=True, weights=new_weights)
    Fd = -RT*np.log(ρd)
    Fd -= Fd[-1]
    ax.plot(edges_to_range(d_edges), Fd, label=forcefield, ls='solid')

    ax.legend()


### Estimate from Metad

In [ ]:
from mg_sampling.tools import read_wsv
from mg_sampling.plot import plot_bias, plot_bias_XY_marginals
from mg_sampling.bias import dissociative_mask, associative_mask
from mg_sampling.bias import bias_fn
from mg_sampling.constants import RT

otypes = ["op"]
forcefields = ["mMg_tip3p", "nMg_tip3p", "mMg_opc", "ecc_spce"]

sims = {
    (ot, ff): MetadSim(f'{PROJECT_DIR}/metad/{ot}_{ff}', f'{ot}_{ff}')
              for ff in forcefields
              for ot in otypes
}

fig, axarr = plt.subplots(nrows=len(forcefields), figsize=(6, 3*len(forcefields)))
_aopts = []
for (ot, ff), sim in sims.items():
    params = read_wsv(f'{sim.path}/bias_params.wsv').iloc[0]
    weights = sim.weights
    mask = sim.x > 0.001
    bias = bias_fn(sim.x, sim.y, *params)
    ΔΔG = -RT * np.log(np.average(np.exp(-bias/RT), weights=weights*mask))
    print(ff, ΔΔG, "kJ/mol")
    def ramp(x, a):
        return a * x
    def loss(a):
        new_bias = bias + ramp(sim.x, a)
        new_weights = weights * np.exp(-ramp(sim.x, a)/RT)
        ΔΔG = -RT * np.log(np.average(np.exp(-new_bias/RT), weights=new_weights*mask))
        return ΔΔG**2
    aopt = minimize_scalar(loss).x
    _aopts.append(aopt)
    new_bias = bias + ramp(sim.x, aopt)
    new_weights = weights * np.exp(-ramp(sim.x, aopt)/RT)
    ΔΔG = -RT * np.log(np.average(np.exp(-new_bias/RT), weights=new_weights*mask))
    print(ff, f"{aopt = } kJ/mol", f"{ΔΔG = } kJ/mol")

In [ ]:
aopts, _aopts

In [ ]:
from mg_sampling.tools import read_wsv
from mg_sampling.plot import plot_bias, plot_bias_XY_marginals
from mg_sampling.bias import dissociative_mask, associative_mask


otypes = ["op"]
forcefields = ["mMg_tip3p", "nMg_tip3p", "mMg_opc", "ecc_spce"]

x_edges, y_edges = np.linspace(0, 2, 61), np.linspace(4, 7, 91)
X_edges, Y_edges = np.linspace(0, 2, 61), np.linspace(-1, 2, 91)

sims = {
    (ot, ff): MetadSim(f'{PROJECT_DIR}/metad/{ot}_{ff}', f'{ot}_{ff}')
              for ff in forcefields
              for ot in otypes
}
for (ot, ff), sim in sims.items():
    params = read_wsv(f'{sim.path}/bias_params.wsv').iloc[0]
    #params = (12, 0.1, -0.3)
    fig = plt.figure(layout='constrained', figsize=(10, 6))
    subfigs = fig.subfigures(1, 2, wspace=0.07)
    ax = subfigs[0].subplots()
    plot_bias(sim, bias_fn, params, bins=(x_edges, y_edges), ax=ax)
    ax1, ax2 = subfigs[1].subplots(nrows=2)
    plot_bias_XY_marginals(sim, bias_fn, params, mask=associative_mask, X_bins=X_edges, Y_bins=Y_edges, axes=(ax1, ax2))

    fig.suptitle(f"{LABELS[ff]}: {LABELS[ot]}", fontsize=16)
    ax1.legend()
    fig.savefig(f"figures/biasfit_{ot}_{ff}.pdf")

### OP-fitted bias on OB

In [ ]:
otypes = ['ob']
forcefields = ["amber_tip3p", "mMg_tip3p", "nMg_tip3p", "mMg_opc"]

sims = {
    (ot, ff): MetadSim(f'{PROJECT_DIR}/metad/{ot}_{ff}', f'{ot}_{ff}')
              for ff in forcefields
              for ot in otypes
}
for (ot, ff), sim in sims.items():
    params = read_wsv(f'{PROJECT_DIR}/metad/op_{ff}/bias_params.wsv').iloc[0]
    fig = plt.figure(layout='constrained', figsize=(10, 6))
    subfigs = fig.subfigures(1, 2, wspace=0.07)
    ax = subfigs[0].subplots()
    plot_bias(sim, params, ax)
    ax1, ax2 = subfigs[1].subplots(nrows=2)
    plot_bias_marginals(sim, params, (ax1, ax2))

    fig.suptitle(f"{LABELS[ff]}: {LABELS[ot]} ({LABELS['op']} fit)", fontsize=16)
    ax1.legend()
    fig.savefig(f"figures/biasfit_opfit_{ot}_{ff}.pdf")